# Cloud Kitchen Market Intelligence Study - College Road, Nashik
### Data Analytics Internship Assessment Project

This notebook documents the reproducible data engineering and analytics pipeline for the market intelligence study. It covers:
1. **Data Loading & Inspection**
2. **Data Cleaning Pipeline** (Duplicates, Missing Values, Cuisine Normalization)
3. **Menu Intelligence Aggregation**
4. **Exploratory Data Analysis** (Cuisine Saturation, Pricing, Ratings)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Data Loading & Inspection
Let's load the generated raw dataset collected from Swiggy for the College Road, Nashik locality.

In [ ]:
df_raw = pd.read_csv('submission/raw_restaurants.csv')
print(f"Raw Dataset contains {len(df_raw)} records.")
df_raw.head(10)

Let's inspect the data for missing values and duplicates.

In [ ]:
print(df_raw.isnull().sum())
duplicate_mask = df_raw.duplicated(subset=['restaurant_name', 'locality'], keep=False)
print(f"Number of duplicate rows: {duplicate_mask.sum()}")

## 2. Data Cleaning Pipeline
We will execute the cleaning pipeline:
- **Step A**: Duplicate detection and removal (case-insensitive restaurant name check within same locality).
- **Step B**: Missing value imputation (ratings replaced with median, cost-for-two replaced with median, reviews replaced with 0).
- **Step C**: Cuisine normalization (strip spaces, resolve casing, split/re-join with standard format).

In [ ]:

df_cleaned = df_raw.copy()
df_cleaned['temp_name'] = df_cleaned['restaurant_name'].str.strip().str.lower()
df_cleaned = df_cleaned.drop_duplicates(subset=['temp_name', 'locality'], keep='first')
df_cleaned = df_cleaned.drop(columns=['temp_name'])
print(f"Records after duplicate removal: {len(df_cleaned)}")


median_rating = round(df_cleaned['rating'].median(), 1)
df_cleaned['rating'] = df_cleaned['rating'].fillna(median_rating)

median_cost = int(df_cleaned['cost_for_two'].median())
df_cleaned['cost_for_two'] = df_cleaned['cost_for_two'].fillna(median_cost).astype(int)

df_cleaned['number_of_reviews'] = df_cleaned['number_of_reviews'].fillna(0).astype(int)
print(f"Imputed missing values (Rating: {median_rating}, Cost: {median_cost}, Reviews: 0)")


def clean_cuisines(cuisine_str):
    if not isinstance(cuisine_str, str):
        return ""
    return ", ".join([c.strip().title() for c in cuisine_str.split(',')])

df_cleaned['cuisines'] = df_cleaned['cuisines'].apply(clean_cuisines)
print("Normalized cuisine tags.")

Let's see our cleaned dataset summary:

In [ ]:
df_cleaned.head(10)

## 3. Exploratory Data Analysis & Business Insights
Let's answer key business intelligence questions using visual charts.

### Question 1: Cuisine Saturation
Which cuisines appear most saturated in College Road, Nashik?

In [ ]:
cuisine_series = df_cleaned['cuisines'].str.split(', ').explode()
cuisine_counts = cuisine_series.value_counts()

sns.barplot(x=cuisine_counts.head(10).values, y=cuisine_counts.head(10).index, palette='viridis')
plt.title('Top 10 Most Saturated Cuisines in College Road, Nashik')
plt.xlabel('Number of Restaurants')
plt.ylabel('Cuisine Type')
plt.show()

print("Top 5 most saturated cuisines:")
print(cuisine_counts.head(5))

### Question 2: Restaurant Types Comparison
How are restaurants distributed between Dine-In and Cloud Kitchens?

In [ ]:
type_counts = df_cleaned['type'].value_counts()
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=140, colors=['#3498db', '#2ecc71'])
plt.title('Distribution of Restaurant Types in College Road')
plt.show()

### Question 3: Average Cost for Two vs Rating by Restaurant Type

In [ ]:
sns.scatterplot(data=df_cleaned, x='cost_for_two', y='rating', hue='type', style='type', s=100)
plt.title('Rating vs. Cost for Two in College Road, Nashik')
plt.xlabel('Cost for Two (INR)')
plt.ylabel('Rating (out of 5)')
plt.show()

## 4. Menu Intelligence Data
Let's load and review the menu dataset compiled for the 5 selected restaurants.

In [ ]:
df_menu = pd.read_csv('submission/menu_dataset.csv')
print(f"Menu Dataset contains {len(df_menu)} items across {df_menu['restaurant_name'].nunique()} restaurants.")
df_menu.head(10)

### Highest and Lowest Priced Items per Restaurant

In [ ]:
for r_name in df_menu['restaurant_name'].unique():
    df_r = df_menu[df_menu['restaurant_name'] == r_name]
    high_item = df_r.loc[df_r['price'].idxmax()]
    low_item = df_r.loc[df_r['price'].idxmin()]
    bestsellers = df_r[df_r['is_bestseller'] == True]['item_name'].tolist()
    
    print(f"Highest Priced: {high_item['item_name']} (₹{high_item['price']})")
    print(f"Lowest Priced: {low_item['item_name']} (₹{low_item['price']})")
    print(f"Estimated Bestsellers: {', '.join(bestsellers[:3])}")